In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import matplotlib as mpl
%matplotlib inline

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/histamine_imaging' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/histamine_imaging' 
savepath_fig.mkdir(parents=True, exist_ok=True)

%config InlineBackend.figure_format = 'retina'

In [ ]:
def factorize_dataset(df, cols_to_factorize_on, new_col_name):
    df[new_col_name] = df.groupby(cols_to_factorize_on,sort=False).ngroup().astype(int)
    return df

def get_stim_params(x, non_stimulus_specs):
    return [e for e in x if e not in non_stimulus_specs]

def average_and_baseline(df):
    averaged = pd.DataFrame(df.groupby(col_names_except(df, ["trial", "value"]), group_keys=False)['value'].mean()).rename(columns={'value': 'averaged_value'}).reset_index()
    baselined_values = averaged.groupby(col_names_except(averaged, ["time", "averaged_value"]), group_keys=False)[['time','averaged_value']].apply(lambda x: subtract_baseline(x, 'averaged_value'))
    baselined = averaged.merge(baselined_values).rename(columns={'value_baselined':'df/f'})
    return baselined.drop(columns='averaged_value')

def col_names_except(df, remove_names):
    all_names = df.columns.to_list()
    names_reduced = [e for e in all_names if e not in remove_names]
    return names_reduced

def subtract_baseline(df, values_column):
    base = df.query("time < 0.0")[values_column].values.mean()
    df = df.copy()
    df["value_baselined"] = df[values_column] - base
    return df

def get_response_range(roi, quantify_max_values, min_range, max_range):
    unique_stim = roi.unique_stimulus.unique()
    response_peak_time = (quantify_max_values.query("unique_stimulus == @unique_stim").time.iloc[0])
    min_time = response_peak_time - min_range
    max_time = response_peak_time + max_range
    
    return roi.loc[(roi['time'] > min_time) & (roi['time'] < max_time)]

In [ ]:
dtypes = {'bg_intensity': 'float64',
 'edge_intensity': 'float64',
 'filename_base': 'O',
 'fly': 'O',
 'genotype': 'O',
 'max_intensity': 'float64',
 'min_intensity': 'float64',
 'name': 'O',
 'roi': 'O',
 'roi_type': 'O',
 'rotation [deg]': 'float64',
 'start pause [sec]': 'float64',
 'stimulus duration [s]': 'float64',
 'time': 'float64',
 'trial': 'O',
 'value': 'float64',
 'velocity [deg/s]': 'float64'}

In [ ]:
# data = pd.read_hdf(inputpath_data / 'preprocessed_histamine_imaging.h5', key='data')
data = pd.read_csv(inputpath_data / 'preprocessed_histamine_imaging.csv', dtype=dtypes)
imaging_specs = ["fly", "genotype", "filename_base"]
roi_specs = ["roi", "roi_type"]
stimulus_specs = ["name", "start pause [sec]", "bg_intensity", "edge_intensity", "rotation [deg]", "velocity [deg/s]",
                  "wavelength [deg]", "max_intensity", "min_intensity", "mode [sq/sin]", "stimulus duration [s]",
                  "pulse length [s]", "intensity",
                  "trial", "time"]
non_stimulus_specs = imaging_specs + roi_specs + ['name', 'trial', 'time', 'value', 'df/f', 'genotype_sort']

In [ ]:
data

In [ ]:
cols_to_factorize_on = ["fly", "filename_base", "roi_type", "roi"]
data_factorized = factorize_dataset(data.reset_index(), cols_to_factorize_on, new_col_name="unique_roi")
data_factorized['genotype'] = data_factorized['genotype'].map({'L1L2_T4T5_HA1m': 'L1L2_T4T5_HA1m'})

palette = sns.color_palette(['#f7b519', '#ee5923'])
lookup_genotype = {'L1L2_T4T5_HA1m': 0}
palette_genotypes = {'L1L2_T4T5_HA1m': '#000000'}

In [ ]:
# Sort according to genotype
df = data_factorized.copy()
df_mapping = pd.DataFrame({'genotype': ['L1L2_T4T5_HA1m', 'L1L2_T4T5_HA1h']})

sort_mapping = df_mapping.reset_index().set_index('genotype')
df['genotype_sort'] = df['genotype'].map(sort_mapping['index'])
df = df.sort_values(['genotype_sort', 'name', 'unique_roi', 'trial', 'time']).reset_index(drop=True)

df['experiment'] = None
df.loc[df['filename_base'].str.contains('edges'), 'experiment'] = 'dir_tuning'
df.loc[df['filename_base'].str.contains('flicker'), 'experiment'] = 'flashes'

df = df[col_names_except(df, 'index')]
df

In [ ]:
### Create lists of all stimulus parameters for each stimulus type ###

stimuli = []
column_names = []

for experiment, dataset in df.groupby('experiment'):
    stimuli.append(experiment)
    column_names.append(dataset.dropna(how='any', axis=1).columns.to_list())
    
stimuli_to_loop = pd.DataFrame({'experiment': stimuli, 'column_names': column_names})
stimuli_to_loop['stim_params'] = stimuli_to_loop['column_names'].apply(lambda row: get_stim_params(row, non_stimulus_specs))

stimuli_to_loop

In [ ]:
### Make separate dataframes per stimulus type, average over trials and baseline subtraction ###

dir_tuning_index = stimuli_to_loop['experiment'].loc[lambda x: x=='dir_tuning'].index[0]
dir_tuning_name = stimuli_to_loop['experiment'][dir_tuning_index]
dir_tuning_data = df.query("experiment == @dir_tuning_name")[stimuli_to_loop.loc[dir_tuning_index, 'column_names']]
dir_tuning_data = average_and_baseline(dir_tuning_data)
dir_tuning_data["contrast"] = np.around(abs(dir_tuning_data["edge_intensity"] - dir_tuning_data["bg_intensity"]) / 255, decimals=2)
dir_tuning_data["contrast"] = dir_tuning_data["contrast"].replace(0.49, 0.5) # calculated 50% contrast rounds to 49% but still want to display it at 50%
dir_tuning_data["polarity"] = np.sign(dir_tuning_data["edge_intensity"] - dir_tuning_data["bg_intensity"])
dir_tuning_data = dir_tuning_data.drop(columns=["edge_intensity", "bg_intensity"])
dir_tuning_data = dir_tuning_data.sort_values(['genotype_sort', 'unique_roi', 'time']).reset_index(drop=True)

flashes_index = stimuli_to_loop['experiment'].loc[lambda x: x=='flashes'].index[0]
flashes_name = stimuli_to_loop['experiment'][flashes_index]
flashes_data = df.query("experiment == @flashes_name")[stimuli_to_loop.loc[flashes_index, 'column_names']]
flashes_data = average_and_baseline(flashes_data)
flashes_data["contrast"] = np.around(abs(flashes_data["max_intensity"] - flashes_data["min_intensity"]) / 255, decimals=2)
flashes_data["contrast"] = flashes_data["contrast"].replace(0.49, 0.5) # calculated 50% contrast rounds to 49% but still want to display it at 50%
flashes_data["polarity"] = np.sign(flashes_data["max_intensity"] - flashes_data["min_intensity"])
flashes_data = flashes_data.drop(columns=["max_intensity", "min_intensity"])
flashes_data = flashes_data.sort_values(['genotype_sort', 'unique_roi', 'time']).reset_index(drop=True)

In [ ]:
all_processed_data = pd.concat([dir_tuning_data, flashes_data])
all_processed_data = all_processed_data.sort_values(['genotype_sort', 'unique_roi', 'time'])

In [ ]:
### Create NEW list of all stimulus parameters for each stimulus type ###

non_stimulus_specs_2 = non_stimulus_specs + ['unique_roi', 'start pause [sec]', 'experiment']
stimuli_2 = []
column_names_2 = []

for experiment, dataset in all_processed_data.groupby('experiment'):
    stimuli_2.append(experiment)
    column_names_2.append(dataset.dropna(how='any', axis=1).columns.to_list())
    
stimuli_to_loop_2 = pd.DataFrame({'experiment': stimuli_2, 'column_names': column_names_2})

stimuli_to_loop_2 = stimuli_to_loop_2.copy()
stimuli_to_loop_2['stim_params'] = stimuli_to_loop_2['column_names'].apply(lambda row: get_stim_params(row, non_stimulus_specs_2))
stimuli_to_loop_2 = stimuli_to_loop_2.set_index('experiment')
stimuli_to_loop_2

In [ ]:
cm_inch = 1/2.54 

df = all_processed_data.query("experiment == 'flashes' and roi_type == 'lam_whole' and genotype == 'L1L2_T4T5_HA1m'")
rows = ['genotype_sort','genotype']

stimuli_loop = []
stim_n_loop = []

for name, stim in df.groupby('experiment'):
    stim_n = stim.groupby(stimuli_to_loop_2.loc[name,'stim_params']).ngroups
    stimuli_loop.append(name)
    stim_n_loop.append(stim_n)

stimuli_to_loop_df = pd.DataFrame({'experiment': stimuli_loop, 'columns_n': stim_n_loop})
cols_n = stimuli_to_loop_df.columns_n.values.sum()

rows_iter = df.groupby(rows, sort=False)
rows_n = rows_iter.ngroups

fig, axs = plt.subplots(rows_n, cols_n, figsize=(8.9 * cm_inch, 1 * cm_inch), sharex=False, sharey=True, squeeze=False, width_ratios=[2.5, 4.5, 10.5],dpi=600)

row_count = 0
col_count = 0

for name_acq, row in rows_iter:
    for name_stim, cols_stim_type in row.groupby('experiment'):
        for name_params, cols_stim_params in cols_stim_type.groupby(stimuli_to_loop_2.loc[name_stim,'stim_params'] + ['roi_type']):
            sns.lineplot(data=cols_stim_params, x='time', y='df/f', units='fly', estimator=None, ax=axs[row_count,col_count], color=palette_genotypes[cols_stim_params.genotype.unique()[0]], alpha = 0.3)
            sns.lineplot(data=cols_stim_params, x='time', y='df/f', ax=axs[row_count,col_count], color=palette_genotypes[cols_stim_params.genotype.unique()[0]], alpha = 1, errorbar=None)

            axs[row_count,col_count].set_ylabel('ΔF/F')
            axs[row_count,col_count].set_xlabel('Time (s)')
            axs[row_count, col_count].set_xlim([-0.5, name_params[2]])
            axs[row_count, col_count].set_ylim([-0.5, 0.6])
            axs[row_count, col_count].set_xticks([0, name_params[2] / 2, name_params[2]])
            axs[row_count, col_count].fill_between([name_params[2] / 2, name_params[2]], 1, -0.5, color='k', alpha=0.05, lw=0)
            axs[row_count, col_count].fill_between([-0.5, 0], 1, -0.5, color='k', alpha=0.05, lw=0)
            if col_count != 0:
                axs[row_count, col_count].yaxis.set_visible(False) 
                axs[row_count, col_count].spines[['left', 'top', 'right']].set_visible(False)
            else:
                axs[row_count, col_count].spines[['top', 'right']].set_visible(False)
                axs[row_count, col_count].text(-0, 0.6, name_acq[1][-4:], color=palette_genotypes[cols_stim_params.genotype.unique()[0]])
            if row_count == 0:
                axs[row_count, col_count].set_title(str(name_params[2]), fontsize=6)
            col_count = col_count + 1
    row_count = row_count + 1
    col_count = 0

figtitle = "L1_L2_HA1m_flicker_responses.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
df = dir_tuning_data.query("experiment == 'dir_tuning' and genotype == 'L1L2_T4T5_HA1m'")
rows = ['polarity' ,'roi_type']

stimuli_loop = []
stim_n_loop = []

for name, stim in df.groupby('experiment'):
    stim_n = stim.groupby(stimuli_to_loop_2.loc[name,'stim_params']).ngroups
    stimuli_loop.append(name)
    stim_n_loop.append(stim_n)

stimuli_to_loop_df = pd.DataFrame({'experiment': stimuli_loop, 'columns_n': stim_n_loop})
cols_n = 8

rows_iter = df.groupby(rows, sort=False)
rows_n = rows_iter.ngroups

fig, axs = plt.subplots(rows_n, cols_n, figsize=(8.9 * cm_inch, 2 * cm_inch), sharex=True, sharey=True, squeeze=False, dpi=600)

row_count = 0
col_count = 0

for name_acq, row in rows_iter:
    for name_stim, cols_stim_type in row.groupby('experiment'):
        for name_params, cols_stim_params in cols_stim_type.groupby(stimuli_to_loop_2.loc[name_stim,'stim_params']):
            
            sns.lineplot(data=cols_stim_params, x='time', y='df/f', units='fly', estimator=None, ax=axs[row_count,col_count], color=palette_genotypes[cols_stim_params.genotype.unique()[0]], alpha = 0.3, lw=1)            
            sns.lineplot(data=cols_stim_params, x='time', y='df/f', ax=axs[row_count,col_count], color=palette_genotypes[cols_stim_params.genotype.unique()[0]], alpha = 1, errorbar=None, lw=1)
            axs[row_count,col_count].set_ylabel('ΔF/F')
            axs[row_count,col_count].set_xlabel('Time (s)')
            axs[row_count, col_count].set_xlim([0, 5.18])
            axs[row_count, col_count].set_ylim([-0.5,0.6])

            if col_count != 0:
                axs[row_count, col_count].yaxis.set_visible(False) 
                axs[row_count, col_count].spines[['left', 'top', 'right']].set_visible(False)
            else:
                axs[row_count, col_count].spines[['top', 'right']].set_visible(False)
                axs[row_count, col_count].text(-1, 0.4, str(cols_stim_params.genotype.unique()[0][-4:]) + '\npolarity ' +  str(cols_stim_params.polarity.unique()[0]) + '\n' + str(cols_stim_params.roi_type.unique()[0]), color=palette_genotypes[cols_stim_params.genotype.unique()[0]])
            if row_count != 1:
                axs[row_count, col_count].xaxis.set_visible(False) 
                axs[row_count, col_count].spines['bottom'].set_visible(False)
            if row_count == 0:
                axs[row_count, col_count].set_title(str(name_params[0]) + " deg", fontsize=6)
            col_count = col_count + 1
    row_count = row_count + 1
    col_count = 0

figtitle = "T4_T5_HA1m_edge_responses.pdf"
plt.savefig(savepath_fig / figtitle)